# REPLACE: Study title

REPLACE: State the research question, selected population, actual coverage and strongest qualification in a short paragraph. This draft authoring template has no executed findings or sample results.

Each insight follows **heading → description → findings summary → chart or table**. The website presents reviewed narrative and evidence outputs with analysis code hidden. The notebook remains the authored source of the analysis and selected presentation.

### Author instructions — keep out of the website

Copy this notebook into `studies/<study-id>/` and use the workspace `.venv` Python 3.11 kernel. Read the template guide before adapting it. Replace every `REPLACE` field and the generic questions, measures, units and cohorts. The three example themes illustrate a table, chronological line and bar chart; they are not required themes or a completed protocol.

The scaffold refuses execution until settings and a pinned-input loader are implemented. There is no fallback data. The loader returns one explicitly defined observation per row, with `group`, `period`, `value` and the configured observation key columns. Validate release/artifact bytes using the actual study lock and configured storage. Website report JSON is not an analytical input.

Keep filters, aggregations, comparisons, findings and visual generation in the cells below each insight. Cell roles and `publish` flags are an authoring convention, not an implemented exporter. A study-local runner/exporter must validate and export executed outputs through Narada's existing contract; do not import this whole notebook or generic HTML into Narada.

The example aggregations are new analysis. For a presentation-only migration, replace them with exact selection of already saved evidence rather than calculating new means, comparisons or conclusions.

In [ ]:
# Configure only after protocol and input selection are explicit.
STUDY_CONFIGURED = False
STUDY_ID = "REPLACE_STUDY_ID"
MEASURE_NAME = "REPLACE_MEASURE"
VALUE_UNIT = "REPLACE_UNIT"
OBSERVATION_UNIT = "REPLACE_OBSERVATION_UNIT"  # eligible session, resolved episode, etc.
OBSERVATION_KEY_COLUMNS = ["observation_id"]  # exact row grain; never silently deduplicate
PERIOD_FORMAT = "%Y-%m"  # replace with the declared chronological bucket format
TIME_AXIS_LABEL = "REPLACE_PERIOD_LABEL"  # calendar/timezone/partial-period context
INPUT_LOCK_NAME = "inputs.lock.json"
INPUT_LOCK_SHA256 = "REPLACE_WITH_LOCK_SHA256"
SCOPE_NOTE = "REPLACE: population, exclusions, support, uncertainty and execution limits"

# Deliberately no synthetic-data switch or automatic input discovery.
if not STUDY_CONFIGURED:
    raise RuntimeError("Draft template: configure the study and its pinned-input loader before Run All.")
required_text = [STUDY_ID, MEASURE_NAME, VALUE_UNIT, OBSERVATION_UNIT, TIME_AXIS_LABEL, SCOPE_NOTE]
if any(not value.strip() or "REPLACE" in value for value in required_text):
    raise RuntimeError("Resolve every study identity, measure, unit and scope placeholder first.")
if not OBSERVATION_KEY_COLUMNS or len(set(OBSERVATION_KEY_COLUMNS)) != len(OBSERVATION_KEY_COLUMNS):
    raise ValueError("Declare distinct observation-key column names for this study's grain.")
if len(INPUT_LOCK_SHA256) != 64 or any(char not in "0123456789abcdef" for char in INPUT_LOCK_SHA256):
    raise ValueError("Pin the exact study input-lock SHA-256.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from abacus_bhishma.paths import load_workspace_paths

paths = load_workspace_paths()
WORKSPACE = paths.workspace_root
STUDY_DIR = WORKSPACE / "studies" / STUDY_ID
input_lock_path = STUDY_DIR / INPUT_LOCK_NAME
input_lock_bytes = input_lock_path.read_bytes()
if hashlib.sha256(input_lock_bytes).hexdigest() != INPUT_LOCK_SHA256:
    raise ValueError("Input-lock bytes do not match the declared identity.")
input_lock = json.loads(input_lock_bytes)

def load_study_observations(input_lock, paths):
    """Implement this study's explicitly pinned loader, never a discovery scan.

    Verify selected dataset/release and every consumed artifact against its lock.
    Resolve storage through paths, preserve available_at and missing/undefined
    event states, and report retrospective eligibility. Return a DataFrame at the
    declared grain. Public data loaders may be called here; study-specific
    analytical selections and calculations belong in the analysis cells below.
    """
    raise NotImplementedError("Implement and verify the pinned loader; no sample data is supplied.")

observations = load_study_observations(input_lock, paths)

In [ ]:
# Fail before generating evidence if these generic example assumptions are unmet.
if not isinstance(observations, pd.DataFrame) or observations.empty:
    raise ValueError("The pinned loader must return a nonempty observations DataFrame.")
required_columns = ["group", "period", "value", *OBSERVATION_KEY_COLUMNS]
missing_columns = sorted(set(required_columns) - set(observations.columns))
if missing_columns:
    raise ValueError(f"Missing declared input columns: {missing_columns}")
if observations[required_columns].isna().any().any():
    raise ValueError("This example requires complete observations. Define and report missingness before adapting it.")
if observations.duplicated(OBSERVATION_KEY_COLUMNS).any():
    raise ValueError("Repeated keys: resolve the analytical grain rather than dropping rows.")
if not observations["group"].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
    raise ValueError("Groups need explicit, nonempty labels.")
if not pd.api.types.is_numeric_dtype(observations["value"]) or pd.api.types.is_bool_dtype(observations["value"]):
    raise ValueError("The measure needs numeric values and explicit units, not booleans or numeric strings.")
if not np.isfinite(observations["value"].to_numpy(dtype=float)).all():
    raise ValueError("Nonfinite measures require an explicit analytical policy.")
observations = observations.copy()
observations["period"] = pd.to_datetime(observations["period"], format=PERIOD_FORMAT, errors="raise")
if observations["period"].nunique() < 2:
    raise ValueError("The line example needs at least two observed periods; choose another display otherwise.")
if observations["group"].nunique() < 2:
    raise ValueError("The category comparison needs at least two groups; choose another question otherwise.")

# Fresh execution output only; never a completed run or an input directory.
execution_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
OUTPUT_DIR = WORKSPACE / "work" / STUDY_ID / execution_id
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

In [ ]:
# Formatting defaults only. This helper calculates no statistics.
INK, GREEN, GOLD, PAPER = "#242a26", "#355c4a", "#a3834f", "#f4f1e9"
plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER,
    "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
})

def evidence_axes(title, subtitle):
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    fig.subplots_adjust(left=0.12, right=0.97, bottom=0.20, top=0.76)
    fig.text(0.035, 0.97, title, fontsize=17, fontweight="bold", ha="left", va="top")
    fig.text(0.055, 0.89, subtitle, fontsize=10, ha="left", va="top")
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    ax.margins(y=0.15)
    return fig, ax

## Population & measurement

What observations are represented, and what values are retained in each group? Rename this theme to match the actual question family.

### What values are observed in each group?

Summarize the declared measure across eligible observations in each group, retaining counts beside means and medians. This example gives every observation one equal weight; replace that assumption when grain, exposure or dependence requires another method. Define missingness, eligibility and units before interpreting the table.

In [ ]:
# All numerical work for this insight is visible here.
group_summary = (
    observations.groupby("group", sort=True, observed=True)["value"]
    .agg(observations="size", mean="mean", median="median")
    .reset_index()
)
group_summary["unit"] = VALUE_UNIT
summary_observation_count = int(group_summary["observations"].sum())
summary_low = float(group_summary["mean"].min())
summary_high = float(group_summary["mean"].max())

In [ ]:
display(Markdown(
    f"**Findings summary.** Across **{summary_observation_count:,} {OBSERVATION_UNIT} observations** "
    f"in {len(group_summary):,} groups, observed group means of {MEASURE_NAME} range from "
    f"**{summary_low:,.2f} to {summary_high:,.2f} {VALUE_UNIT}**. "
    "The table retains group support and medians. These descriptive point estimates "
    "do not establish a causal difference or executable advantage. " + SCOPE_NOTE
))

In [ ]:
# Save full precision; round only a displayed copy. Plain DataFrame display
# uses the locked notebook environment without optional Styler/Jinja2.
group_summary.to_csv(OUTPUT_DIR / "group-summary.csv", index=False)
display(group_summary.round({"mean": 2, "median": 2}))


## Change over time

How does the selected measure vary across comparable chronological periods? Rename periods and theme to reflect the actual time convention.

### How does the measure vary over time?

Calculate a mean in each observed period, retain eligible-observation counts and connect the chronologically ordered estimates. Identify partial and missing periods explicitly. This is a descriptive connected line, not a fitted regression or significance test; connecting points does not make their coverage comparable.

In [ ]:
# Keep chronology explicit; never sort time periods by result size.
period_summary = (
    observations.groupby("period", sort=True, observed=True)["value"]
    .agg(observations="size", mean="mean")
    .reset_index()
    .sort_values("period")
)
first_period = period_summary.iloc[0]
last_period = period_summary.iloc[-1]
period_low = float(period_summary["mean"].min())
period_high = float(period_summary["mean"].max())

In [ ]:
display(Markdown(
    f"**Findings summary.** Across {len(period_summary):,} observed periods, mean {MEASURE_NAME} "
    f"ranges from **{period_low:,.2f} to {period_high:,.2f} {VALUE_UNIT}**. "
    f"The first displayed period ({first_period['period']:%Y-%m-%d}) has "
    f"{int(first_period['observations']):,} observations and the last "
    f"({last_period['period']:%Y-%m-%d}) has {int(last_period['observations']):,}. "
    "Changes may reflect coverage, composition or support; this line alone does not "
    "demonstrate a persistent trend. " + SCOPE_NOTE
))

In [ ]:
period_summary.to_csv(OUTPUT_DIR / "period-trend.csv", index=False)
fig, ax = evidence_axes(
    f"{MEASURE_NAME} over time",
    f"Descriptive means per {OBSERVATION_UNIT}; labels give eligible n",
)
ax.plot(period_summary["period"], period_summary["mean"], marker="o", color=GREEN, linewidth=2)
for row in period_summary.itertuples(index=False):
    ax.annotate(f"n={row.observations:,}", (row.period, row.mean), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=9)
ax.set(xlabel=TIME_AXIS_LABEL, ylabel=f"Mean {MEASURE_NAME} ({VALUE_UNIT})")
ax.set_xticks(period_summary["period"], period_summary["period"].dt.strftime(PERIOD_FORMAT))
fig.autofmt_xdate()
fig.savefig(OUTPUT_DIR / "period-trend.png", dpi=160, bbox_inches="tight", pad_inches=0.2)
plt.show()

## Context comparisons

How does the measure differ across declared categories? Fix category definitions and the relevant baseline in the study protocol.

### How do the category averages compare?

Compare the same measured quantity across declared groups using bars starting at zero and eligible support labels. This example reuses the computed group summary with the same population. Replace the comparison and baseline with the actual hypothesis; retain uncertainty and sparse-support qualifications when required.

In [ ]:
# Reuse exactly the population and estimates calculated for the table insight.
category_comparison = group_summary[["group", "observations", "mean"]].copy()
category_low_row = category_comparison.loc[category_comparison["mean"].idxmin()]
category_high_row = category_comparison.loc[category_comparison["mean"].idxmax()]
category_range = float(category_high_row["mean"] - category_low_row["mean"])

In [ ]:
display(Markdown(
    f"**Findings summary.** Observed category means span **{category_range:,.2f} {VALUE_UNIT}**: "
    f"{category_low_row['group']} has {category_low_row['mean']:,.2f} "
    f"(n={int(category_low_row['observations']):,}) and {category_high_row['group']} "
    f"has {category_high_row['mean']:,.2f} (n={int(category_high_row['observations']):,}). "
    "This span is descriptive and selected after observing the categories. It is not "
    "an adjusted contrast, a significance result or a trading recommendation. " + SCOPE_NOTE
))

In [ ]:
category_comparison.to_csv(OUTPUT_DIR / "category-comparison.csv", index=False)
fig, ax = evidence_axes(
    f"{MEASURE_NAME} by category",
    f"Same eligible {OBSERVATION_UNIT} population as the group table; descriptive means",
)
x = np.arange(len(category_comparison))
ax.bar(x, category_comparison["mean"], color=GREEN, width=0.6)
ax.set_xticks(x, category_comparison["group"], rotation=25, ha="right")
ax.axhline(0, color=GOLD, linewidth=1)
for at, row in zip(x, category_comparison.itertuples(index=False)):
    offset = 6 if row.mean >= 0 else -14
    ax.annotate(f"{row.mean:,.2f}\nn={row.observations:,}", (at, row.mean), xytext=(0, offset), textcoords="offset points", ha="center", fontsize=9)
ax.set(xlabel="Declared group", ylabel=f"Mean {MEASURE_NAME} ({VALUE_UNIT})")
fig.savefig(OUTPUT_DIR / "category-comparison.png", dpi=160, bbox_inches="tight", pad_inches=0.2)
plt.show()

In [ ]:
# Accounting checks do not replace numerical tests or scientific review.
if summary_observation_count != len(observations):
    raise ValueError("Group evidence lost or duplicated eligible observations.")
if int(period_summary["observations"].sum()) != len(observations):
    raise ValueError("Chronological evidence lost or duplicated eligible observations.")
if not period_summary["period"].is_monotonic_increasing:
    raise ValueError("Time evidence is not chronologically ordered.")
if not category_comparison.equals(group_summary[["group", "observations", "mean"]]):
    raise ValueError("The category chart and its stated source table differ.")

# Add study-specific known-answer/edge-case tests and independent notebook-to-web
# comparisons for every selected value, finding, table and image before publishing.
# Bind source cells, lock, environment, command and output hashes in a run receipt.
# These template checks are not a research seal or publication approval.

### Author delivery checklist — keep out of the website

- Replace example questions, descriptions, measures and claims with the adopted protocol. Remove insights that do not answer a useful question.
- Restart the kernel and Run All in fresh staging. Record exact source, inputs, environment, command, exclusions and verification outcomes.
- Give each insight a stable ID and one heading, description, executed findings summary and primary chart/table. Export appropriate captions, alt text, units, denominators and uncertainty/support notes.
- Compare selected website values and conclusions with the executed notebook and underlying evidence. Preserve contrary and inconclusive results.
- Export only selected prose, typed data and raster figures through a study-local exporter. Narada must not execute the notebook or expose setup/debug output.
- Keep audit metadata in retained source/run/revision evidence. Keep essential scope and interpretation visible; do not recreate the removed notebook publication-record directory.
- New results, methods or interpretation require a new run. Layout-only migration preserves saved values and meaning in a new presentation revision with a new publication ID.
- Review the notebook before website preview. Template adoption and local checks do not authorize a GitHub push or Vercel deployment.